# 41 — Explainability

Global permutation importance and local one-feature reference-substitution sensitivity on the frozen ARGOS V1 model. These are predictive explanations, **not causal effects**.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec
from argos.config.robustness import load_robustness_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
(paths.root / "reports" / "robustness").mkdir(parents=True, exist_ok=True)
(paths.root / "data" / "m5").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "models" / "m5").mkdir(parents=True, exist_ok=True)

m4_gate = paths.docs / "decisions" / "M4_GATE.md"
assert m4_gate.exists() and "M4 ✅ CLOSED" in m4_gate.read_text(encoding="utf-8"), (
    "M4 no está formalmente cerrado. No iniciar M5."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
M5_SPEC = load_robustness_spec(paths.configs / "robustness.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

assert GOLD_SPEC.schema_version == M5_SPEC.core.benchmark_gold_schema
assert MODEL_SPEC.schema_version == M5_SPEC.core.benchmark_model_schema
{
    "gold_schema": GOLD_SPEC.schema_version,
    "model_schema": MODEL_SPEC.schema_version,
    "m5_schema": M5_SPEC.schema_version,
    "benchmark_model": M5_SPEC.core.benchmark_model_name,
    "benchmark_calibration": M5_SPEC.core.benchmark_calibration,
    "consumed_test_year": M5_SPEC.governance.consumed_test_year,
    "fresh_oot_year": M5_SPEC.governance.fresh_oot_year,
}

assert (paths.reports / "robustness" / "40_validation.parquet").exists(), "Ejecuta 40 primero."
from argos.robustness.explainability import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


'm5-41-permutation-local-sensitivity-v2'

In [2]:
from argos.robustness.explainability import BUILD_IMPLEMENTATION, build_explainability

print("implementation:", BUILD_IMPLEMENTATION)

result = build_explainability(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    robustness_spec=M5_SPEC,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
)
result.global_importance.head(20)


implementation: m5-41-permutation-local-sensitivity-v2


,feature,family,baseline_pr_auc,importance_mean_pr_auc_drop,importance_std,repeats,sample_rows
0,codigo_de_categoria_principal,contract,0.032537,0.012207,0.001595,3,50000
1,t0_month,contract,0.032537,0.007767,0.000374,3,50000
2,prior_entity_contract_count,historical,0.032537,0.007466,0.001392,3,50000
3,justificacion_modalidad_de_contratacion,contract,0.032537,0.004976,0.001191,3,50000
4,prior_supplier_entity_addition_event_count,historical,0.032537,0.002935,0.000658,3,50000
5,modalidad_de_contratacion,contract,0.032537,0.002469,0.000444,3,50000
6,departamento,contract,0.032537,0.001751,0.000477,3,50000
7,ciudad,contract,0.032537,0.001475,0.001532,3,50000
8,sector,contract,0.032537,0.001241,0.001526,3,50000
9,tipo_de_contrato,contract,0.032537,0.001040,0.000278,3,50000


In [3]:
result.family_importance


,family,feature_count,total_pr_auc_drop,mean_pr_auc_drop,max_pr_auc_drop
0,contract,23,0.033577,0.001460,0.012207
1,historical,12,0.007592,0.000633,0.007466
2,competition,6,0.000325,0.000054,0.000112


In [4]:
result.local_sensitivity.head(30)


,contract_id,t0,target,feature,family,reference_value,base_raw_score,perturbed_raw_score,raw_score_delta,base_calibrated_score,perturbed_calibrated_score,calibrated_score_delta,interpretation
0,CO1.PCCNTR.6681039,2024-08-27,0,t0_year,contract,2024.0,0.986427,0.986427,0.000000,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
1,CO1.PCCNTR.6681039,2024-08-27,0,t0_month,contract,9.0,0.986427,0.972669,0.013759,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
2,CO1.PCCNTR.6681039,2024-08-27,0,t0_day_of_week,contract,3.0,0.986427,0.986427,0.000000,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
3,CO1.PCCNTR.6681039,2024-08-27,0,t0_quarter,contract,3.0,0.986427,0.986427,0.000000,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
4,CO1.PCCNTR.6681039,2024-08-27,0,departamento,contract,Distrito Capital de Bogotá,0.986427,0.986427,0.000000,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
5,CO1.PCCNTR.6681039,2024-08-27,0,ciudad,contract,No Definido,0.986427,0.986427,0.000000,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
6,CO1.PCCNTR.6681039,2024-08-27,0,orden,contract,Territorial,0.986427,0.986427,0.000000,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
7,CO1.PCCNTR.6681039,2024-08-27,0,sector,contract,Servicio Público,0.986427,0.956352,0.030075,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
8,CO1.PCCNTR.6681039,2024-08-27,0,rama,contract,Ejecutivo,0.986427,0.986427,0.000000,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...
9,CO1.PCCNTR.6681039,2024-08-27,0,entidad_centralizada,contract,Centralizada,0.986427,0.986427,0.000000,0.130938,0.130938,0.0,one-feature reference substitution; sensitivit...


In [5]:
result.validation


,check_id,category,blocking,passed,observed,requirement
0,explainability_uses_validation_only,governance,True,True,"{""consumed_test_year"": 2025, ""max_t0"": ""2024-1...",feature attribution is derived from developmen...
1,global_importance_complete,explainability,True,True,"{""expected"": 41, ""features_reported"": 41}",every frozen V1 feature has a finite permutati...
2,global_importance_minimum_reported,explainability,True,True,"{""features"": 41, ""minimum"": 10}",global explainability artifact contains a nont...
3,local_sensitivity_complete,explainability,True,True,"{""cases"": 25, ""features"": 41}",local sensitivity is persisted for the frozen ...
4,explainability_model_hash_preserved,lineage,True,True,"{""calibrator_sha256"": ""95e8e506f92b191f7315a1f...",explainability uses the exact frozen M4 V1 mod...


In [6]:
blocking = result.validation.loc[result.validation["blocking"]]
print("41 ✅ PASS" if result.ready_for_next_notebook else "41 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook


41 ✅ PASS
Blocking checks: 5/5
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m5_explainability.json
